In [3]:
import torch
import os
import gc
from diffusers import StableDiffusionXLPipeline
from diffusers.loaders import LoraLoaderMixin
from safetensors.torch import load_file

# 1. تنظيف الذاكرة
gc.collect()
torch.cuda.empty_cache()

# 2. المسارات
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, "..")) if "notebooks" in current_dir else current_dir

base_model_path = os.path.join(base_dir, "models", "base", "animagine-xl-3.1.safetensors")
lora_path = os.path.join(base_dir, "models", "lora", "my_anime_character_lora.safetensors")
output_fused_dir = os.path.join(base_dir, "models", "base", "animagine_fused_full")

print("... جاري تحميل النموذج الأساسي ...")
pipe = StableDiffusionXLPipeline.from_single_file(
    base_model_path,
    torch_dtype=torch.float16,
    use_safetensors=True
)

print("... جاري قراءة ملف الـ LoRA يدوياً ...")
if lora_path.endswith(".safetensors"):
    state_dict = load_file(lora_path)
else:
    state_dict = torch.load(lora_path, map_location="cpu")

print("... جاري تفكيك ودمج أوزان الـ UNet والـ Text Encoders ...")

def apply_lora_to_model(model, state_dict, prefix, lora_scale=1.0):
    """دالة لدمج أوزان الـ LoRA مباشرة في أوزان الموديل الأساسية بدون PEFT"""
    model_state = model.state_dict()
    updates = 0
    
    # البحث عن الأزواج lora_down و lora_up
    for key in list(state_dict.keys()):
        if prefix in key and "lora_down.weight" in key:
            up_key = key.replace("lora_down.weight", "lora_up.weight")
            alpha_key = key.replace("lora_down.weight", "alpha")
            
            if up_key in state_dict:
                down_weight = state_dict[key].to(dtype=torch.float32)
                up_weight = state_dict[up_key].to(dtype=torch.float32)
                
                # حساب المقياس (Alpha Scaling)
                rank = down_weight.shape[0]
                alpha = state_dict[alpha_key].item() if alpha_key in state_dict else rank
                scale = (alpha / rank) * lora_scale
                
                # حساب التغير في الأوزان W_delta = (Up @ Down) * scale
                if len(up_weight.shape) == 4: # Conv2d
                    delta = torch.mm(up_weight.flatten(1), down_weight.flatten(1)).view(up_weight.shape) * scale
                else: # Linear
                    delta = torch.matmul(up_weight, down_weight) * scale
                
                # إيجاد الاسم المطابق في الموديل الأصلي
                target_key = key.replace(prefix + "_", "").replace(".lora_down.weight", ".weight")
                target_key = target_key.replace(".", "_", target_key.count(".") - 1) if target_key not in model_state else target_key
                
                # مطابقة مفاتيح diffusers
                for m_key in model_state.keys():
                    clean_m_key = m_key.replace(".", "_")
                    clean_target_key = target_key.replace(".", "_")
                    if clean_m_key == clean_target_key or m_key.endswith(target_key):
                        model_state[m_key] += delta.to(device=model_state[m_key].device, dtype=model_state[m_key].dtype)
                        updates += 1
                        break
                        
    print(f"  - تم دمج {updates} طبقة لـ {prefix}")

# 1. دمج أوزان الـ UNet
apply_lora_to_model(pipe.unet, state_dict, prefix="lora_unet", lora_scale=1.0)

# 2. دمج أوزان الـ Text Encoder 1
if hasattr(pipe, "text_encoder") and pipe.text_encoder is not None:
    apply_lora_to_model(pipe.text_encoder, state_dict, prefix="lora_te1", lora_scale=1.0)
    apply_lora_to_model(pipe.text_encoder, state_dict, prefix="lora_te", lora_scale=1.0)

# 3. دمج أوزان الـ Text Encoder 2 (خاص بـ SDXL)
if hasattr(pipe, "text_encoder_2") and pipe.text_encoder_2 is not None:
    apply_lora_to_model(pipe.text_encoder_2, state_dict, prefix="lora_te2", lora_scale=1.0)

print(f"... جاري حفظ النموذج المدمج بالكامل في: {output_fused_dir} ...")
pipe.save_pretrained(output_fused_dir, safe_serialization=True)

# تنظيف الذاكرة
del pipe
gc.collect()
torch.cuda.empty_cache()

print("✅ تم دمج وحفظ النموذج الكامل (UNet + Text Encoders) بنجاح وبدون أخطاء!")

... جاري تحميل النموذج الأساسي ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


... جاري قراءة ملف الـ LoRA يدوياً ...
... جاري تفكيك ودمج أوزان الـ UNet والـ Text Encoders ...
  - تم دمج 0 طبقة لـ lora_unet
  - تم دمج 0 طبقة لـ lora_te1
  - تم دمج 0 طبقة لـ lora_te
  - تم دمج 192 طبقة لـ lora_te2
... جاري حفظ النموذج المدمج بالكامل في: c:\Users\OrbaTechs\Documents\Visual_Studio_18\cartoon-generator\models\base\animagine_fused_full ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ تم دمج وحفظ النموذج الكامل (UNet + Text Encoders) بنجاح وبدون أخطاء!


In [20]:
import torch
import gc
import os
from diffusers import AnimateDiffSDXLPipeline, MotionAdapter, AutoencoderKL, DPMSolverMultistepScheduler
from diffusers.utils import export_to_gif
from diffusers.utils.logging import disable_progress_bar

disable_progress_bar()

# 1. Clean memory
gc.collect()
torch.cuda.empty_cache()

# 2. Paths
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, "..")) if "notebooks" in current_dir else current_dir

fused_model_dir = os.path.join(base_dir, "models", "base", "animagine_fused_full")

print("[INFO] Loading VAE in float32...")
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", 
    torch_dtype=torch.float32
)

print("[INFO] Loading Motion Adapter...")
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-sdxl-beta", torch_dtype=torch.float16)

print("[INFO] Loading fused pipeline...")
pipe = AnimateDiffSDXLPipeline.from_pretrained(
    fused_model_dir,
    motion_adapter=adapter,
    vae=vae,
    torch_dtype=torch.float16,
    use_safetensors=True
)

# تثبيت المجدول الممتاز
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, 
    use_karras_sigmas=True
)

# 3. Memory management
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()

# 4. نفس البرومبت الناجح تماماً بدون أي تغيير في بنية الشخصية
trigger_word = "mycustomboy" 
prompt = f"masterpiece, best quality, sharp focus, clean 2D anime style, cowboy shot, centered character, consistent facial features, stable structure, {trigger_word}, short black hair, red eyes, 1boy, wearing green hooded adventurer cloak, detailed dark stone dungeon background, stone floor, glowing wall torches, atmospheric lighting"

# أضفنا فقط كلمات إزالة الضباب والدخان الخلفي في الـ Negative لتنقية الجو العام
negative_prompt = "wide shot, long shot, distant figure, full body wide shot, small character in frame, close-up, portrait, upper body only, lowres, bad anatomy, deformed, distorted, stretched, morphing face, multiple faces, duplicate facial features, shifting eyes, blurry, noise, grainy, pixelated, jpeg artifacts, grid lines, tiling artifacts, smoke, haze, fog, atmospheric dust, airborne particles, blurry background"

print("[INFO] Generating final clean sequence...")

with torch.cuda.amp.autocast(dtype=torch.float16):
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_frames=16,
        height=768,
        width=768,
        guidance_scale=9.0,
        num_inference_steps=40,
        generator=torch.Generator("cuda").manual_seed(42)
    )

frames = output.frames[0]

# 5. Save result
output_gif_path = os.path.join(base_dir, "character_lora_fixed_test.gif")
export_to_gif(frames, output_gif_path, fps=8)

print(f"[SUCCESS] Generation complete! Check the clean result at: {output_gif_path}")

[INFO] Loading VAE in float32...
[INFO] Loading Motion Adapter...
[INFO] Loading fused pipeline...


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

There are modules in UNetMotionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


[INFO] Generating final clean sequence...


C:\Users\OrbaTechs\AppData\Local\Temp\ipykernel_18072\1395764207.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (93 > 77). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (93 > 77). Running this sequence through the model will result in indexing errors


  0%|          | 0/40 [00:00<?, ?it/s]

[SUCCESS] Generation complete! Check the clean result at: c:\Users\OrbaTechs\Documents\Visual_Studio_18\cartoon-generator\character_lora_fixed_test.gif
